# Cleaning the auditing ground-truth data (DSO 576, Module 6)

**Source:** `TheFinAI/Herculean` on Hugging Face, file `auditing_env/XBRL/mathematical_reasoning.csv` (CC BY 4.0), downloaded 2026-09-30, dataset commit `52d3ecb`.

**One row:** one DQC (Data Quality Committee) rule that fired on one XBRL filing, with the rule's message in the free-text column `assertion.detail`.

**How to run:** open this notebook from the project root (the folder that contains `herculean_data/`), then **Restart → Run All**. The original CSV is only read, never written.

**Cleaning rules, written before applying them**
1. The original file and the DataFrame `df` stay unchanged. All work happens on a copy called `clean`.
2. The reported value and the calculated value are inside `assertion.detail`. Extract them with one pattern per template (types 126, 117, 15). A value that cannot be extracted stays empty and is counted.
3. Convert amounts to numbers and dates to datetime. Count failed conversions. Never drop a row because of a failed conversion.
4. Split `period` into `period_start` / `period_end`. A single date is a point in time (instant), so `period_start` stays empty.
5. Standardize `entity.name` (trim, collapse whitespace including non-breaking spaces). Keep identifiers as text: CIK padded to 10 digits, accession numbers untouched.
6. Type 15 (sign check) has no expected value by design, so `calculated_value` and `rule_element_id` stay empty. They are not filled.
7. Do not delete any row. Rows that look repeated are kept when their message text differs (different statement or dimension).
8. Never "correct" a reported or calculated value: every row is already a DQC error. Suspicious rows are marked with flags (`sign_flip`, `period_after_filing`).

## 1. Load and inspect (checklist items 2, 3, 4)

In [1]:
import os
import re
import difflib
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
pd.options.display.float_format = lambda x: f'{x:,.6f}'.rstrip('0').rstrip('.')

SRC = 'herculean_data/auditing_env/XBRL/mathematical_reasoning.csv'
df = pd.read_csv(SRC, dtype={'entity.cik': str})   # the original; never modified below

print("shape:", df.shape)
print(df.dtypes)
print("\nmissing values in the original file:", int(df.isnull().sum().sum()))
print("exact duplicate rows:", int(df.duplicated().sum()))

shape: (362, 20)
assertion.code               object
assertion.detail             object
assertion.run-date           object
assertion.severity           object
assertion.source             object
assertion.type                int64
entity.cik                   object
entity.name                  object
report.accepted-timestamp    object
report.accession             object
report.base-taxonomy         object
report.creation-software     object
report.document-type         object
report.entry-url             object
report.filing-date           object
report.filing-year            int64
report.id                     int64
report.sec-url               object
report.sic-code               int64
folder_name                  object
dtype: object

missing values in the original file: 0
exact duplicate rows: 0


In [2]:
print(df['assertion.type'].value_counts().sort_index())
print("\ndistinct assertion.code:", df['assertion.code'].nunique())
print("\nassertion.severity:")
print(df['assertion.severity'].value_counts())
print("\ndistinct filings (report.accession):", df['report.accession'].nunique())
print("most rows for one filing:", int(df.groupby('report.accession').size().max()))

assertion.type
15     116
117    136
126    110
Name: count, dtype: int64

distinct assertion.code: 61

assertion.severity:
assertion.severity
ERROR    362
Name: count, dtype: int64

distinct filings (report.accession): 158
most rows for one filing: 28


## 2. Extract the values from the message text (item 4, 7)

In [3]:
NUM = r'-?[\d,]+(?:\.\d+)?'      # allows decimals such as -0.0554 and thousands separators

def to_num(s):
    return s.replace(',', '') if s else None

def parse_detail(row):
    text = row['assertion.detail']
    t = row['assertion.type']
    out = {}

    m = re.match(r'^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{3})', text)
    out['timestamp'] = m.group(1) if m else None

    # prefix:Concept in any namespace (us-gaap, srt, or a company prefix such as nmrk)
    m = re.search(r'\b([a-z][\w-]*):([A-Z]\w+)', text)
    out['concept_prefix'] = m.group(1) if m else None
    out['concept'] = m.group(2) if m else None

    m = re.search(r'(?:Total period|Period)\s*:\s*([\d-]+(?:\s*to\s*[\d-]+)?)', text)
    out['period'] = m.group(1).strip() if m else None

    # whole value up to the end of the line (units can contain "/", e.g. USD/bbl)
    m = re.search(r'Unit(?: of Total)?[ \t]*:[ \t]*([^\r\n]+?)[ \t]*$', text, re.M)
    out['unit'] = m.group(1) if m else None

    m = re.search(r'Rule Element Id\s*:\s*(\d+)', text)
    out['rule_element_id'] = m.group(1) if m else None

    m = re.search(r'Rule version:\s*(.+?)\s*$', text)
    out['rule_version'] = m.group(1) if m else None

    out['comparison_type'] = None
    out['reported_value'] = None
    out['calculated_value'] = None

    if t == 126:      # total vs sum of child components in the calculation linkbase
        out['comparison_type'] = 'calculation_sum'
        m = re.search(rf'Total Value\s*:\s*({NUM})', text)
        out['reported_value'] = to_num(m.group(1)) if m else None
        m = re.search(rf'sum of these child components is ({NUM})', text)
        out['calculated_value'] = to_num(m.group(1)) if m else None
    elif t == 117:    # total vs dimensional breakdown
        out['comparison_type'] = 'dimensional_breakdown'
        m = re.search(rf'value of ({NUM}) is not equal to the dimensional breakdown of ({NUM})', text)
        if m:
            out['reported_value'] = to_num(m.group(1))
            out['calculated_value'] = to_num(m.group(2))
    elif t == 15:     # sign check: there is no expected value by design
        out['comparison_type'] = 'sign_check'
        m = re.search(rf'has a value of ({NUM}) which is', text)
        out['reported_value'] = to_num(m.group(1)) if m else None

    return pd.Series(out)

parsed = df.apply(parse_detail, axis=1)
clean = pd.concat([df.copy(), parsed], axis=1)
n0 = len(clean)

fields = ['timestamp', 'concept_prefix', 'concept', 'period', 'unit', 'rule_element_id',
          'rule_version', 'reported_value', 'calculated_value']
print("missing extracted fields, by assertion.type:")
print(clean[fields].isnull().groupby(clean['assertion.type']).sum().T)
print("\nconcept_prefix counts:")
print(clean['concept_prefix'].value_counts())
print("\nunit counts:")
print(clean['unit'].value_counts())

missing extracted fields, by assertion.type:
assertion.type    15   117  126
timestamp           0    0    0
concept_prefix      0    0    0
concept             0    0    0
period              0    0    0
unit                0    0    0
rule_element_id   116    0    0
rule_version        0    0    0
reported_value      0    0    0
calculated_value  116    0    0

concept_prefix counts:
concept_prefix
us-gaap    350
ncmi         6
srt          3
nmrk         1
orcl         1
king         1
Name: count, dtype: int64

unit counts:
unit
USD        330
shares      15
pure        12
bbl          4
USD/bbl      1
Name: count, dtype: int64


## 3. Types and dates (item 4)

In [4]:
for c in ['reported_value', 'calculated_value']:
    before = int(clean[c].notnull().sum())
    clean[c] = pd.to_numeric(clean[c], errors='coerce')
    after = int(clean[c].notnull().sum())
    print(f"{c}: non-null before={before}, after={after}, failed={before - after}")

for c in ['report.filing-date', 'report.accepted-timestamp', 'assertion.run-date']:
    conv = pd.to_datetime(clean[c], errors='coerce')
    print(f"{c}: failed conversions = {int(conv.isnull().sum())}")
    clean[c] = conv
clean['timestamp'] = pd.to_datetime(clean['timestamp'], format='%Y-%m-%d %H:%M:%S,%f', errors='coerce')
print("timestamp (from the message text): failed =", int(clean['timestamp'].isnull().sum()))

has_range = clean['period'].str.contains(' to ', na=False)
parts = clean['period'].str.split(r'\s*to\s*', expand=True)
if parts.shape[1] < 2:
    parts[1] = None
clean['period_start'] = pd.to_datetime(parts[0].where(has_range), errors='coerce')
clean['period_end'] = pd.to_datetime(parts[1].where(has_range, parts[0]), errors='coerce')
print(f"\nperiod: {int(has_range.sum())} ranges, {int((~has_range).sum())} single dates (instant)")

reported_value: non-null before=362, after=362, failed=0
calculated_value: non-null before=246, after=246, failed=0
report.filing-date: failed conversions = 0
report.accepted-timestamp: failed conversions = 0
assertion.run-date: failed conversions = 0
timestamp (from the message text): failed = 0

period: 237 ranges, 125 single dates (instant)


## 4. Standardize text (item 5)

In [5]:
before = clean['entity.name']
after = before.str.strip().str.replace(r'\s+', ' ', regex=True)   # \s also matches the non-breaking space
changed = before != after
print("entity.name changed by trim / whitespace collapse:", int(changed.sum()))
for i in clean.index[changed][:3]:
    print(f"  {before[i]!r} -> {after[i]!r}")
clean['entity.name'] = after

clean['entity.cik'] = clean['entity.cik'].astype(str).str.zfill(10)   # CIK is a 10-digit identifier
print("\nunique entity.name before/after:", df['entity.name'].nunique(), "/", clean['entity.name'].nunique())
print("unique entity.cik:", clean['entity.cik'].nunique())
print("max names per cik after cleaning:", int(clean.groupby('entity.cik')['entity.name'].nunique().max()))

entity.name changed by trim / whitespace collapse: 7
  'BARNWELL INDUSTRIES,\xa0INC.' -> 'BARNWELL INDUSTRIES, INC.'
  'BARNWELL INDUSTRIES,\xa0INC.' -> 'BARNWELL INDUSTRIES, INC.'
  'BARNWELL INDUSTRIES,\xa0INC.' -> 'BARNWELL INDUSTRIES, INC.'

unique entity.name before/after: 125 / 125
unique entity.cik: 125
max names per cik after cleaning: 1


## 5. Duplicates (item 6)

In [6]:
clean['body'] = clean['assertion.detail'].str.split(r'\]\s', n=1, regex=True).str[1]   # message without the timestamp
row_key = ['report.accession', 'assertion.code', 'body']
weak_key = ['report.accession', 'assertion.code', 'concept_prefix', 'concept', 'period', 'reported_value']

print("exact duplicate rows (original):", int(df.duplicated().sum()))
print("repeated report.id:", int(df['report.id'].duplicated().sum()), "(a filing has several assertions, so expected)")
print("repeated on a short key", weak_key, ":", int(clean.duplicated(subset=weak_key).sum()))
print("repeated on the final row key", row_key, ":", int(clean.duplicated(subset=row_key).sum()))

exact duplicate rows (original): 0
repeated report.id: 204 (a filing has several assertions, so expected)
repeated on a short key ['report.accession', 'assertion.code', 'concept_prefix', 'concept', 'period', 'reported_value'] : 18
repeated on the final row key ['report.accession', 'assertion.code', 'body'] : 0


In [7]:
# why the short-key repeats are NOT duplicates: show how the message text differs (one pair per type)
def show_diff(i, j):
    a, b = clean.loc[i, 'body'], clean.loc[j, 'body']
    sm = difflib.SequenceMatcher(None, a, b, autojunk=False)
    for tag, a0, a1, b0, b1 in sm.get_opcodes():
        if tag != 'equal':
            print(f"  [{tag}] A: {a[max(0, a0-30):a1+30]!r}")
            print(f"        B: {b[max(0, b0-30):b1+30]!r}")

dups = clean[clean.duplicated(subset=weak_key, keep=False)]
print("rows in short-key repeat groups:", len(dups), "| by type:", dups['assertion.type'].value_counts().to_dict())
seen = set()
for k, g in dups.groupby(weak_key):
    t = g['assertion.type'].iloc[0]
    if t in seen:
        continue
    seen.add(t)
    i, j = g.index[:2]
    print(f"\n--- type {t}: group of {len(g)}, rows {i} vs {j} ---")
    show_diff(i, j)

rows in short-key repeat groups: 26 | by type: {117: 17, 15: 9}

--- type 117: group of 5, rows 70 vs 71 ---
  [insert] A: 'In the statement 100100012 - Statement - Statements'
        B: 'In the statement 100180020 - Statement - Statements '
  [insert] A: 'In the statement 100100012 - Statement - Statements o'
        B: 'In the statement 100180020 - Statement - Statements of '
  [delete] A: 'In the statement 100100012 - Statement - Statements of I'
        B: 'In the statement 100180020 - Statement - Statements of I'
  [replace] A: 'ment - Statements of Income - APC the concept us-gaap:Revenue'
        B: 'ment - Statements of Income - GPC the concept us-gaap:Revenue'
  [delete] A: 'h is comprised of the members:\n\tProductAndServiceOtherMember \t --> 755,000,000 \t --> srt:ProductOrServiceAxis=us-gaap:ProductAndServiceOtherMember \n\tRetailMember \t --> 14,084,00'
        B: 'h is comprised of the members:\n\tRetailMember \t --> 14,084,00'
  [insert] A: 'Axis=so:WholesaleNonAffiliat

## 6. Missing data (item 7)

In [8]:
print("missing extracted fields, by assertion.type:")
print(clean[fields].isnull().groupby(clean['assertion.type']).sum().T)
print("\nperiod_start missing:", int(clean['period_start'].isnull().sum()),
      "| single-date (instant) rows:", int((~has_range).sum()), "-> structural, a point in time has no start")
print("type 15 rows:", int((clean['assertion.type'] == 15).sum()),
      "-> rule_element_id and calculated_value are empty by design, not filled")

missing extracted fields, by assertion.type:
assertion.type    15   117  126
timestamp           0    0    0
concept_prefix      0    0    0
concept             0    0    0
period              0    0    0
unit                0    0    0
rule_element_id   116    0    0
rule_version        0    0    0
reported_value      0    0    0
calculated_value  116    0    0

period_start missing: 125 | single-date (instant) rows: 125 -> structural, a point in time has no start
type 15 rows: 116 -> rule_element_id and calculated_value are empty by design, not filled


## 7. Suspicious values: flag, never correct (item 8)

In [9]:
clean['period_after_filing'] = clean['period_end'] > clean['report.filing-date']
clean['sign_flip'] = (clean['calculated_value'] == -clean['reported_value']) & (clean['reported_value'] != 0)
same = (clean['calculated_value'] == clean['reported_value']) & clean['calculated_value'].notnull()

print("sign_flip (calculated == -reported):", int(clean['sign_flip'].sum()))
print("period_after_filing:", int(clean['period_after_filing'].sum()))
print("reported == calculated (every row is a DQC error, so expected 0):", int(same.sum()))
print("period_start after period_end:", int((clean['period_start'] > clean['period_end']).sum()))

print("\nsign-flip rows:")
print(clean.loc[clean['sign_flip'], ['assertion.type', 'concept', 'period', 'reported_value', 'calculated_value']].to_string())
print("\nperiod_after_filing rows:")
print(clean.loc[clean['period_after_filing'], ['assertion.type', 'concept', 'period', 'report.filing-date', 'reported_value']].to_string())

sign_flip (calculated == -reported): 9
period_after_filing: 2
reported == calculated (every row is a DQC error, so expected 0): 0
period_start after period_end: 0

sign-flip rows:
     assertion.type                                                  concept                    period  reported_value  calculated_value
1               126               NetCashProvidedByUsedInFinancingActivities  2020-10-01 to 2021-09-30         707,500          -707,500
6               126               NetCashProvidedByUsedInInvestingActivities  2021-04-01 to 2021-12-31   2,771,000,000    -2,771,000,000
9               126               NetCashProvidedByUsedInOperatingActivities  2021-07-01 to 2021-12-31          41,260           -41,260
21              126               NetCashProvidedByUsedInInvestingActivities  2021-11-01 to 2022-01-31      -3,395,318         3,395,318
45              126                           OtherOperatingIncomeExpenseNet  2022-01-01 to 2022-09-30         -32,002            32,00

## 8. Reconcile and check denominators (items 10, 11)

In [10]:
t_before = df['assertion.type'].value_counts().sort_index()
t_after = clean['assertion.type'].value_counts().sort_index()
assert len(clean) == len(df), "row count changed"
assert (t_before == t_after).all(), "rows by type changed"
assert not clean.duplicated(subset=row_key).any(), "final row key is not unique"

print("rows: original", len(df), "-> cleaned", len(clean), "| dropped:", len(df) - len(clean))
print(pd.DataFrame({'original': t_before, 'cleaned': t_after}))

n_cmp = int(clean['calculated_value'].notnull().sum())
print(f"\nsign_flip rate = {int(clean['sign_flip'].sum())}/{n_cmp} = {clean['sign_flip'].sum() / n_cmp:.1%}"
      f"  (denominator = rows that have a calculated_value; {int(clean['calculated_value'].isnull().sum())} type-15 rows have none)")

rows: original 362 -> cleaned 362 | dropped: 0
                original  cleaned
assertion.type                   
15                   116      116
117                  136      136
126                  110      110

sign_flip rate = 9/246 = 3.7%  (denominator = rows that have a calculated_value; 116 type-15 rows have none)


## 9. Record checks (item 12)
The expected values and the reason for each row were written from the raw message text before the output was looked at.

In [11]:
exp66 = dict(reported=-614011, calculated=614011)
TEXT_117 = '255,500,000 is not equal to the dimensional breakdown of 142,400,000'
idx117 = df.index[df['assertion.detail'].str.contains(TEXT_117, regex=False)]
print("rows matching the type-117 text:", len(idx117))

expected = {
    0:   dict(prefix='us-gaap', concept='NetCashProvidedByUsedInOperatingActivities', period='2020-10-01 to 2021-09-30',
              reported=-1313536, calculated=-1212736, why='baseline type 126: Total Value vs sum of child components'),
    13:  dict(prefix='nmrk', concept='NetIncomeLossAfterAdjustmentsForNonCashAndNonoperatingItems', period=None,
              reported=-300897000, calculated=1655371000, why='company-specific prefix; the first version of the pattern missed it'),
    188: dict(prefix='us-gaap', concept='DefinedBenefitPlanAssumptionsUsedCalculatingNetPeriodicBenefitCostDiscountRate',
              period='2023-01-01 to 2023-12-31', reported=-0.0554, calculated=None,
              why='type 15, decimal value, period ends after filing date, no calculated value by design'),
    66:  dict(prefix='us-gaap', concept='OperatingExpenses', period='2022-01-01 to 2022-09-30',
              reported=exp66['reported'], calculated=exp66['calculated'], why='sign-flip row'),
}
if len(idx117):
    expected[idx117[0]] = dict(prefix='us-gaap', concept='RevenueFromContractWithCustomerIncludingAssessedTax',
                               period='2020-10-01 to 2020-12-31', reported=255500000, calculated=142400000,
                               why='type 117: total vs dimensional breakdown')

def num_ok(actual, exp):
    if exp is None:
        return pd.isnull(actual)
    return (not pd.isnull(actual)) and np.isclose(float(actual), float(exp))

pat = (r'(?:The value of the concept \S+ of [-\d,.]+|sum of these child components is [-\d,.]+|'
       r'value of [-\d,.]+ is not equal to the dimensional breakdown of [-\d,.]+|has a value of [-\d,.]+)')

for i, e in expected.items():
    if i not in clean.index:
        print(f"\n--- row {i}: not in this file, skipped ---")
        continue
    r = clean.loc[i]
    print(f"\n--- row {i} | {e['why']} ---")
    print("original text :", re.findall(pat, df.loc[i, 'assertion.detail']))
    checks = {
        'prefix': r['concept_prefix'] == e['prefix'],
        'concept': r['concept'] == e['concept'],
        'period': (e['period'] is None) or (r['period'] == e['period']),
        'reported': num_ok(r['reported_value'], e['reported']),
        'calculated': num_ok(r['calculated_value'], e['calculated']),
    }
    print("actual        :", r['concept_prefix'], r['concept'], r['period'], r['reported_value'], r['calculated_value'])
    print("match         :", checks, "->", "ALL MATCH" if all(checks.values()) else "MISMATCH")
    if i == 188:
        print("period_after_filing:", bool(r['period_after_filing']), "(expected True)")

rows matching the type-117 text: 1

--- row 0 | baseline type 126: Total Value vs sum of child components ---
original text : ['The value of the concept us-gaap:NetCashProvidedByUsedInOperatingActivities of -1,313,536', 'sum of these child components is -1,212,736.']
actual        : us-gaap NetCashProvidedByUsedInOperatingActivities 2020-10-01 to 2021-09-30 -1313536.0 -1212736.0
match         : {'prefix': True, 'concept': True, 'period': True, 'reported': True, 'calculated': True} -> ALL MATCH

--- row 13 | company-specific prefix; the first version of the pattern missed it ---
original text : ['The value of the concept nmrk:NetIncomeLossAfterAdjustmentsForNonCashAndNonoperatingItems of -300,897,000', 'sum of these child components is 1,655,371,000.']
actual        : nmrk NetIncomeLossAfterAdjustmentsForNonCashAndNonoperatingItems 2021-01-01 to 2021-12-31 -300897000.0 1655371000.0
match         : {'prefix': True, 'concept': True, 'period': True, 'reported': True, 'calculated': True} ->

In [12]:
# the unit fix: the old pattern cut "USD/bbl" down to "USD". Look at the raw end of the message for row 103.
if 103 in df.index:
    print(df.loc[103, 'assertion.detail'][-250:])
    print("\nextracted unit:", clean.loc[103, 'unit'])

a negated label.

The properties of this us-gaap:DerivativeAverageForwardPrice fact are:
Period :2021-12-31
Dimensions : us-gaap:DerivativeInstrumentRiskAxis=crk:NaturalGasContractsThreeMember
Unit : USD/bbl
Rule version: 16.0.11 - crk-20211231.htm 4

extracted unit: USD/bbl


## 10. Save (the original file is untouched)

In [13]:
os.makedirs('output', exist_ok=True)
out = clean.drop(columns=['body'])
out.to_csv('output/auditing_clean.csv', index=False)
print("saved output/auditing_clean.csv", out.shape)

saved output/auditing_clean.csv (362, 34)


## 11. Check a second implementation (optional: only runs if the Codex output exists)

In [14]:
codex_path = 'output_codex/mathematical_reasoning_clean.csv'
if not os.path.exists(codex_path):
    print("no Codex output found, skipping")
else:
    import warnings
    a = pd.read_csv('output/auditing_clean.csv', dtype={'entity.cik': str})
    b = pd.read_csv(codex_path, dtype={'entity.cik': str})
    b = b.rename(columns={'concept_name': 'concept'})   # same field, Codex's name

    print("shapes (mine, Codex):", a.shape, b.shape)
    print("only in mine :", sorted(set(a.columns) - set(b.columns)))
    print("only in Codex:", sorted(set(b.columns) - set(a.columns)))
    assert len(a) == len(b), "different row counts, compare by key instead"
    aligned = (a['report.accession'] == b['report.accession']).all() and (a['assertion.detail'] == b['assertion.detail']).all()
    print("rows aligned (same accession and detail text, row by row):", aligned)
    assert aligned, "row order differs, merge on a key before comparing"

    STRICT = {'entity.cik'}   # identifiers are compared as text so a missing leading zero is caught

    def equal(x, y, col):
        sx, sy = x.fillna('').astype(str).str.strip(), y.fillna('').astype(str).str.strip()
        if col in STRICT:
            return sx.eq(sy)
        xn, yn = pd.to_numeric(x, errors='coerce'), pd.to_numeric(y, errors='coerce')
        if x.notnull().any() and xn.notnull().sum() == x.notnull().sum() and yn.notnull().sum() == y.notnull().sum():
            return pd.Series(np.isclose(xn.to_numpy(float), yn.to_numpy(float), equal_nan=True), index=x.index)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            xd, yd = pd.to_datetime(x, errors='coerce'), pd.to_datetime(y, errors='coerce')
        if x.notnull().any() and xd.notnull().sum() == x.notnull().sum() and yd.notnull().sum() == y.notnull().sum():
            return (xd == yd) | (xd.isnull() & yd.isnull())
        return sx.eq(sy)

    common = [c for c in a.columns if c in b.columns]
    bad = {c: ~equal(a[c], b[c], c) for c in common}
    bad = {c: m for c, m in bad.items() if m.any()}
    print(f"\ncommon columns: {len(common)} | columns with differences: {len(bad)}")
    for c, m in bad.items():
        print(f"\n{c}: {int(m.sum())} rows differ")
        print(pd.DataFrame({'mine': a.loc[m, c], 'codex': b.loc[m, c]}).head(3).to_string())

shapes (mine, Codex): (362, 34) (362, 31)
only in mine : ['comparison_type', 'rule_element_id', 'rule_version']
only in Codex: []
rows aligned (same accession and detail text, row by row): True

common columns: 31 | columns with differences: 1

period_start: 125 rows differ
   mine       codex
14  NaN  2021-12-31
15  NaN  2021-12-31
16  NaN  2021-12-31


In [15]:
usd = clean['unit'].eq('USD')
print(clean['unit'].value_counts().to_string())
print("\nUSD rows:", int(usd.sum()), "| other units:", int((~usd).sum()))
print("non-USD rows by assertion.type:", clean.loc[~usd, 'assertion.type'].value_counts().to_dict())

unit
USD        330
shares      15
pure        12
bbl          4
USD/bbl      1

USD rows: 330 | other units: 32
non-USD rows by assertion.type: {15: 32}
